In [88]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
!pip install optuna
import optuna

In [89]:
df=pd.read_csv('/content/bengaluru_house_prices.csv')
df.head()

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00


In [90]:
df.describe()

,bath,balcony,price
count,13247.000000,12711.000000,13320.000000
mean,2.692610,1.584376,112.565627
std,1.341458,0.817263,148.971674
min,1.000000,0.000000,8.000000
25%,2.000000,1.000000,50.000000
50%,2.000000,2.000000,72.000000
75%,3.000000,2.000000,120.000000
max,40.000000,3.000000,3600.000000


In [91]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13320 entries, 0 to 13319
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   area_type     13320 non-null  object 
 1   availability  13320 non-null  object 
 2   location      13319 non-null  object 
 3   size          13304 non-null  object 
 4   society       7818 non-null   object 
 5   total_sqft    13320 non-null  object 
 6   bath          13247 non-null  float64
 7   balcony       12711 non-null  float64
 8   price         13320 non-null  float64
dtypes: float64(3), object(6)
memory usage: 936.7+ KB


In [92]:
df.isnull().sum()

,0
area_type,0
availability,0
location,1
size,16
society,5502
total_sqft,0
bath,73
balcony,609
price,0


In [93]:
df['society'].value_counts()

,count
society,
GrrvaGr,80
PrarePa,76
Sryalan,59
Prtates,59
GMown E,56
...,...
SLtalry,1
Rencyes,1
DiaveEn,1


In [94]:
df= df.drop(['society'], axis=1)

In [95]:
df['bath'].fillna(df['bath'].median(), inplace=True)

/tmp/ipykernel_1227/3321556508.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['bath'].fillna(df['bath'].median(), inplace=True)


In [96]:
df['balcony'].value_counts()

,count
balcony,
2.0,5113
1.0,4897
3.0,1672
0.0,1029


In [97]:
df['balcony'].fillna(df['balcony'].median(), inplace=True)

/tmp/ipykernel_1227/2345094989.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['balcony'].fillna(df['balcony'].median(), inplace=True)


In [98]:
df['size'].value_counts()

,count
size,
2 BHK,5199
3 BHK,4310
4 Bedroom,826
4 BHK,591
3 Bedroom,547
1 BHK,538
2 Bedroom,329
5 Bedroom,297
6 Bedroom,191


In [99]:
df['size'].fillna('2 BHK', inplace=True)

/tmp/ipykernel_1227/2227372260.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['size'].fillna('2 BHK', inplace=True)


In [100]:
df['location'].value_counts()

,count
location,
Whitefield,540
Sarjapur Road,399
Electronic City,302
Kanakpura Road,273
Thanisandra,234
...,...
3rd Stage Raja Rajeshwari Nagar,1
Chuchangatta Colony,1
"Electronic City Phase 1,",1


In [101]:
df['location'].fillna('Whitefield', inplace=True)

/tmp/ipykernel_1227/447724410.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['location'].fillna('Whitefield', inplace=True)


In [102]:
df.head()

,area_type,availability,location,size,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,1200,2.0,1.0,51.00


In [103]:
df['availability'].unique()

array(['19-Dec', 'Ready To Move', '18-May', '18-Feb', '18-Nov', '20-Dec',
       '17-Oct', '21-Dec', '19-Sep', '20-Sep', '18-Mar', '20-Feb',
       '18-Apr', '20-Aug', '18-Oct', '19-Mar', '17-Sep', '18-Dec',
       '17-Aug', '19-Apr', '18-Jun', '22-Dec', '22-Jan', '18-Aug',
       '19-Jan', '17-Jul', '18-Jul', '21-Jun', '20-May', '19-Aug',
       '18-Sep', '17-May', '17-Jun', '21-May', '18-Jan', '20-Mar',
       '17-Dec', '16-Mar', '19-Jun', '22-Jun', '19-Jul', '21-Feb',
       'Immediate Possession', '19-May', '17-Nov', '20-Oct', '20-Jun',
       '19-Feb', '21-Oct', '21-Jan', '17-Mar', '17-Apr', '22-May',
       '19-Oct', '21-Jul', '21-Nov', '21-Mar', '16-Dec', '22-Mar',
       '20-Jan', '21-Sep', '21-Aug', '14-Nov', '19-Nov', '15-Nov',
       '16-Jul', '15-Jun', '17-Feb', '20-Nov', '20-Jul', '16-Sep',
       '15-Oct', '15-Dec', '16-Oct', '22-Nov', '15-Aug', '17-Jan',
       '16-Nov', '20-Apr', '16-Jan', '14-Jul'], dtype=object)

In [104]:
df['area_type'].unique()

array(['Super built-up  Area', 'Plot  Area', 'Built-up  Area',
       'Carpet  Area'], dtype=object)

In [105]:
df['BHK/bedrom']=df['size'].str.split(' ').str[1]
df['size']=df['size'].str.split(' ').str[0]

In [106]:
df['BHK/bedrom'].unique()

array(['BHK', 'Bedroom', 'RK'], dtype=object)

In [107]:
import pandas as pd
df['size']=pd.to_numeric(df['size'], errors='coerce').astype('Int64')

In [108]:
import re
def sqft_to_numeric(value):
    if pd.isna(value):
        return np.nan

    text = str(value).strip()

    # Handles values such as "2100 - 2850"
    if "-" in text:
        parts = text.split("-")
        try:
            return (float(parts[0].strip()) + float(parts[1].strip())) / 2
        except:
            return np.nan

    # Handles values such as "34.46Sq. Meter"
    match = re.search(r"[-+]?\d*\.?\d+", text)

    if match:
        number = float(match.group())

        # Convert common non-square-foot units to sqft
        lower = text.lower()

        if "sq. meter" in lower or "sqm" in lower:
            number *= 10.7639
        elif "sq. yards" in lower or "sq yard" in lower:
            number *= 9
        elif "perch" in lower:
            number *= 272.25
        elif "acre" in lower:
            number *= 43560
        elif "guntha" in lower:
            number *= 1089

        return number

    return np.nan


df["total_sqft_numeric"] = df["total_sqft"].apply(sqft_to_numeric)

In [109]:
df= df.drop(['total_sqft'], axis=1)

In [110]:
df

,area_type,availability,location,size,bath,balcony,price,BHK/bedrom,total_sqft_numeric
0,Super built-up Area,19-Dec,Electronic City Phase II,2,2.0,1.0,39.07,BHK,1056.0
1,Plot Area,Ready To Move,Chikka Tirupathi,4,5.0,3.0,120.00,Bedroom,2600.0
2,Built-up Area,Ready To Move,Uttarahalli,3,2.0,3.0,62.00,BHK,1440.0
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3,3.0,1.0,95.00,BHK,1521.0
4,Super built-up Area,Ready To Move,Kothanur,2,2.0,1.0,51.00,BHK,1200.0
...,...,...,...,...,...,...,...,...,...
13315,Built-up Area,Ready To Move,Whitefield,5,4.0,0.0,231.00,Bedroom,3453.0
13316,Super built-up Area,Ready To Move,Richards Town,4,5.0,2.0,400.00,BHK,3600.0
13317,Built-up Area,Ready To Move,Raja Rajeshwari Nagar,2,2.0,1.0,60.00,BHK,1141.0
13318,Super built-up Area,18-Jun,Padmanabhanagar,4,4.0,1.0,488.00,BHK,4689.0


In [111]:
df.loc[df['availability'].str.contains('-'), 'availability'] = 'Future Date'
df['availability']= df['availability'].replace({'Immediate Possession': 'Ready To Move'})

In [112]:
df

,area_type,availability,location,size,bath,balcony,price,BHK/bedrom,total_sqft_numeric
0,Super built-up Area,Future Date,Electronic City Phase II,2,2.0,1.0,39.07,BHK,1056.0
1,Plot Area,Ready To Move,Chikka Tirupathi,4,5.0,3.0,120.00,Bedroom,2600.0
2,Built-up Area,Ready To Move,Uttarahalli,3,2.0,3.0,62.00,BHK,1440.0
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3,3.0,1.0,95.00,BHK,1521.0
4,Super built-up Area,Ready To Move,Kothanur,2,2.0,1.0,51.00,BHK,1200.0
...,...,...,...,...,...,...,...,...,...
13315,Built-up Area,Ready To Move,Whitefield,5,4.0,0.0,231.00,Bedroom,3453.0
13316,Super built-up Area,Ready To Move,Richards Town,4,5.0,2.0,400.00,BHK,3600.0
13317,Built-up Area,Ready To Move,Raja Rajeshwari Nagar,2,2.0,1.0,60.00,BHK,1141.0
13318,Super built-up Area,Future Date,Padmanabhanagar,4,4.0,1.0,488.00,BHK,4689.0


In [113]:
df.duplicated().sum()

np.int64(658)

In [114]:
df.drop_duplicates(inplace=True)

In [115]:
df

,area_type,availability,location,size,bath,balcony,price,BHK/bedrom,total_sqft_numeric
0,Super built-up Area,Future Date,Electronic City Phase II,2,2.0,1.0,39.07,BHK,1056.0
1,Plot Area,Ready To Move,Chikka Tirupathi,4,5.0,3.0,120.00,Bedroom,2600.0
2,Built-up Area,Ready To Move,Uttarahalli,3,2.0,3.0,62.00,BHK,1440.0
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3,3.0,1.0,95.00,BHK,1521.0
4,Super built-up Area,Ready To Move,Kothanur,2,2.0,1.0,51.00,BHK,1200.0
...,...,...,...,...,...,...,...,...,...
13314,Super built-up Area,Ready To Move,Green Glen Layout,3,3.0,3.0,112.00,BHK,1715.0
13315,Built-up Area,Ready To Move,Whitefield,5,4.0,0.0,231.00,Bedroom,3453.0
13316,Super built-up Area,Ready To Move,Richards Town,4,5.0,2.0,400.00,BHK,3600.0
13317,Built-up Area,Ready To Move,Raja Rajeshwari Nagar,2,2.0,1.0,60.00,BHK,1141.0


In [116]:
# Strip any leading/trailing spaces from location
df['location'] = df['location'].apply(lambda x: x.strip())

# Find locations with 10 or fewer data points
location_stats = df['location'].value_counts()
location_stats_less_than_10 = location_stats[location_stats <= 10]

# Replace those rare locations with 'Other'
df['location'] = df['location'].apply(lambda x: 'Other' if x in location_stats_less_than_10 else x)

print(f"New number of unique locations: {df['location'].nunique()}")

New number of unique locations: 233


In [117]:
df.dropna(inplace=True)

In [118]:
df = df.drop(['area_type', 'availability', 'BHK/bedrom'], axis=1)

In [119]:
df = df[~(df.total_sqft_numeric / df['size'] < 300)]

In [120]:
df['price_per_sqft'] = df['price'] * 100000 / df['total_sqft_numeric']
def remove_pps_outliers(df):
    df_out = pd.DataFrame()
    for key, subdf in df.groupby('location'):
        m = np.mean(subdf.price_per_sqft)
        st = np.std(subdf.price_per_sqft)
        # Keep data within 1 standard deviation of the mean
        reduced_df = subdf[(subdf.price_per_sqft > (m - st)) & (subdf.price_per_sqft <= (m + st))]
        df_out = pd.concat([df_out, reduced_df], ignore_index=True)
    return df_out

df = remove_pps_outliers(df)

In [121]:
df.drop(['price_per_sqft'], axis=1, inplace=True)

In [122]:
df = df[df.bath < df['size'] + 2]

In [123]:
X = df.drop('price', axis=1)
y = df['price']

In [125]:
numerical_cols = ['size', 'bath', 'balcony', 'total_sqft_numeric']
categorical_cols = ['location']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), categorical_cols)
    ])

X_processed = preprocessor.fit_transform(X)

print(f"Processed feature matrix shape: {X_processed.shape}")

Processed feature matrix shape: (9717, 236)


In [126]:
X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

X_train shape: (7773, 236)
X_test shape: (1944, 236)


In [127]:
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping

def objective(trial):
    n_layers = trial.suggest_int('n_layers', 1, 5)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)

    model = keras.Sequential()
    model.add(keras.Input(shape=(X_train.shape[1],)))

    for i in range(n_layers):
        num_units = trial.suggest_int(f'n_units_l{i}', 32, 256, step=32)
        dropout_rate = trial.suggest_float(f'dropout_l{i}', 0.1, 0.4)

        model.add(layers.Dense(num_units, activation='relu'))
        model.add(layers.Dropout(dropout_rate))

    model.add(layers.Dense(1))

    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(loss='mse', optimizer=optimizer, metrics=['mae'])

    early_stopping = EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    )

    history = model.fit(
        X_train, y_train,
        validation_split=0.2,
        epochs=30,
        batch_size=64,
        callbacks=[early_stopping],
        verbose=0 #
    )

    return min(history.history['val_loss'])

In [128]:
import optuna
study = optuna.create_study(direction='minimize', study_name="House_Price_NN")

study.optimize(objective, n_trials=15)

print("\n--- Optuna Study Complete ---")
print(f"Best validation MSE: {study.best_value}")
print("Best hyperparameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

[I 2026-08-14 22:25:48,125] A new study created in memory with name: House_Price_NN
[I 2026-08-14 22:26:08,504] Trial 0 finished with value: 6110.92724609375 and parameters: {'n_layers': 3, 'learning_rate': 0.00021451590943245887, 'n_units_l0': 128, 'dropout_l0': 0.26941726509872255, 'n_units_l1': 96, 'dropout_l1': 0.11844953769475962, 'n_units_l2': 128, 'dropout_l2': 0.38876292469669416}. Best is trial 0 with value: 6110.92724609375.
[I 2026-08-14 22:26:22,750] Trial 1 finished with value: 6031.2109375 and parameters: {'n_layers': 3, 'learning_rate': 0.0007131280587632299, 'n_units_l0': 64, 'dropout_l0': 0.26032246583495045, 'n_units_l1': 128, 'dropout_l1': 0.11531426939583882, 'n_units_l2': 64, 'dropout_l2': 0.220803425618046}. Best is trial 1 with value: 6031.2109375.
[I 2026-08-14 22:26:34,179] Trial 2 finished with value: 5742.26513671875 and parameters: {'n_layers': 5, 'learning_rate': 0.0006661064782348806, 'n_units_l0': 96, 'dropout_l0': 0.1333518263283002, 'n_units_l1': 96, 'd


--- Optuna Study Complete ---
Best validation MSE: 3390.185302734375
Best hyperparameters:
  n_layers: 4
  learning_rate: 0.003073204580428636
  n_units_l0: 192
  dropout_l0: 0.20933118057439393
  n_units_l1: 192
  dropout_l1: 0.38763412387600826
  n_units_l2: 224
  dropout_l2: 0.1583612727052037
  n_units_l3: 32
  dropout_l3: 0.39919683197872496


In [129]:
from sklearn.metrics import r2_score

best_params = study.best_params

final_model = keras.Sequential()
final_model.add(keras.Input(shape=(X_train.shape[1],)))

for i in range(best_params['n_layers']):
    final_model.add(layers.Dense(best_params[f'n_units_l{i}'], activation='relu'))
    final_model.add(layers.Dropout(best_params[f'dropout_l{i}']))

final_model.add(layers.Dense(1))

final_model.compile(
    loss='mse',
    optimizer=keras.optimizers.Adam(learning_rate=best_params['learning_rate']),
    metrics=['mae']
)

history = final_model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=64,
    callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)],
    verbose=1
)

y_pred = final_model.predict(X_test)

r2 = r2_score(y_test, y_pred)
print(f"Test R2 Score: {r2:.4f}")

Epoch 1/50
98/98 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - loss: 5604.6670 - mae: 41.8829 - val_loss: 5312.8823 - val_mae: 29.1402
Epoch 2/50
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 33303.6328 - mae: 35.1788 - val_loss: 4543.0620 - val_mae: 27.3297
Epoch 3/50
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 72680.3984 - mae: 35.5383 - val_loss: 4092.9084 - val_mae: 25.0067
Epoch 4/50
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 29135.0664 - mae: 34.0723 - val_loss: 3797.4814 - val_mae: 24.5268
Epoch 5/50
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 112456.0625 - mae: 37.4149 - val_loss: 5372.2959 - val_mae: 35.3752
Epoch 6/50
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 36792.3633 - mae: 36.4281 - val_loss: 4721.9946 - val_mae: 26.2926
Epoch 7/50
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 11468.5361 - mae: 32.7440 - val_loss: 4504.7671 - val_mae: 24.2824
Epoch 8/50
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 5582.2983 - mae: 31.0839 - val_loss: 3988.8599 - val_mae: 22.8213

In [132]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score,mean_absolute_percentage_error
import numpy as np

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
mape = mean_absolute_percentage_error(y_test, y_pred)

print("--- Model Performance Showcase ---")
print(f"R-squared (R2):          {r2:.4f}")
print(f"Root Mean Squared Error: {rmse:.4f} Lakhs")
print(f"Mean Absolute Error:     {mae:.4f} Lakhs")
print(f"Mean Absolute Percentage Error: {mape:.4f}")

--- Model Performance Showcase ---
R-squared (R2):          0.8255
Root Mean Squared Error: 35.6843 Lakhs
Mean Absolute Error:     18.4518 Lakhs
Mean Absolute Percentage Error: 0.2026
